# 論文實作筆記 1：隨機切分的成績為什麼會虛高？

重現 DTA-GNN（Özsari 等，*SoftwareX* 2026, 34, 102671）想解決的問題：用 ChEMBL 資料建出單一標靶的親和力資料集，
比較隨機切分與骨架切分，並用它的洩漏稽核量化骨架重疊。

- 標靶：ChEMBL `CHEMBL1827`（人類 cGMP 專一的 3',5'-環單磷酸雙酯酶 5A）。**這個標靶在看到任何結果之前就定好。**
- 切分：70 / 10 / 20，seed 0–4。模型：隨機森林與 Tanimoto 核 SVR，特徵為 ECFP4（2048 位元）。
- 執行環境：免費 Colab，**不需要 GPU**。最後一格會存出 `results.csv`。
- 文章：<https://517justin.github.io/cheminfo-blog/>（《電腦也會化學》論文實作筆記）
- 內容採用 CC BY 4.0 授權，作者 Cheninformatics。資料來自 ChEMBL（CC BY-SA 3.0）。

In [ ]:
# 1. 安裝與環境記錄（版本會寫進文章）
# RDKit 釘在 2026.9.1：在作者的測試環境中，2026.9.2 的 Murcko 骨架函式會丟出 "bad any cast"，
# 而 DTA-GNN 的骨架切分把這個錯誤吞掉，結果沒有任何分子被分到任何一組。
!pip -q install dta-gnn "rdkit==2026.9.1"
import sys, platform, importlib.metadata as md_
import numpy as np, pandas as pd, sklearn, rdkit
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)
print("scikit-learn", sklearn.__version__, "| rdkit", rdkit.__version__, "| dta-gnn", md_.version("dta-gnn"))
try:
    import torch
    print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())
except Exception as e:
    print("torch 未安裝", e)

In [ ]:
# 2. 用 DTA-GNN 從 ChEMBL 建資料集（走網路 API，會顯示進度；請記下耗時）
import time
from dta_gnn.pipeline import Pipeline

TARGET = "CHEMBL1827"
t0 = time.time()
raw = Pipeline(source_type="web").build_dta(target_ids=[TARGET], split_method="random")
print(f"筆數 {len(raw)}，耗時 {time.time() - t0:.0f} 秒")
raw[["molecule_chembl_id", "smiles", "label"]].head()

In [ ]:
# 3. 資料檢查：重複結構與 pChEMBL 分布
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

df0 = raw[["molecule_chembl_id", "smiles", "label"]].dropna().reset_index(drop=True)
mols = [Chem.MolFromSmiles(s) for s in df0.smiles]
keep = [m is not None for m in mols]
print("無法解析的 SMILES：", len(keep) - sum(keep))
df0 = df0[keep].reset_index(drop=True)
mols = [m for m in mols if m is not None]

# 去掉立體化學後相同的結構（不同 ChEMBL ID 的立體異構物會被當成不同分子）
flat = [Chem.MolToSmiles(m, isomericSmiles=False) for m in mols]
assert df0.molecule_chembl_id.is_unique
print("分子數", len(df0), "| 去立體化學後不重複", len(set(flat)))
print(df0.label.describe().round(2))

In [ ]:
# 4. 特徵與骨架
import numpy as np
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(m) for m in mols]
X = np.array([list(fp) for fp in fps], dtype=np.uint8)
y = df0.label.values.astype(float)
scaf = np.array([MurckoScaffold.MurckoScaffoldSmiles(s) for s in df0.smiles])  # 若這行出錯，請檢查 RDKit 版本
flat = np.array(flat)
print("骨架種類", len(set(scaf)), "| 空骨架（無環）分子數", int((scaf == "").sum()))

In [ ]:
# 5. 主實驗：兩種切分 × 兩種模型 × 5 個 seed
from rdkit import DataStructs
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from dta_gnn.splits import split_random, split_cold_drug_scaffold
from dta_gnn.audits import audit_scaffold_leakage

def tanimoto_kernel(A, B):
    A = A.astype(np.float32); B = B.astype(np.float32)
    inter = A @ B.T
    return inter / (A.sum(1, keepdims=True) + B.sum(1, keepdims=True).T - inter)

def run(keep, tag):
    rows = []
    d0 = df0[keep].reset_index(drop=True)
    Xk, yk, sk, fk = X[keep], y[keep], scaf[keep], flat[keep]
    fpk = [f for f, k in zip(fps, keep) if k]
    for seed in range(5):
        for name in ["random", "scaffold"]:
            d = d0[["molecule_chembl_id", "smiles", "label"]].copy()
            d["split"] = ""
            if name == "random":
                d, *_ = split_random(d, test_size=0.2, val_size=0.1, seed=seed)
            else:
                d = split_cold_drug_scaffold(d, test_size=0.2, val_size=0.1, seed=seed)
            # split_random 會重新排列並重設索引，所以一律用 ChEMBL ID 對回原本的列
            pos = d0.reset_index().set_index("molecule_chembl_id")["index"]
            tr = pos.loc[d.molecule_chembl_id[d.split == "train"]].values
            te = pos.loc[d.molecule_chembl_id[d.split == "test"]].values
            assert set(tr).isdisjoint(te)
            # 防呆：切分不能是空的（RDKit 版本不對時，骨架切分會悄悄失敗）
            assert len(tr) > 0 and len(te) > 0 and (d.split != "").all(), f"{name} split 有分子沒被分到任何一組"
            aud = audit_scaffold_leakage(d.iloc[tr], d.iloc[te])
            seen = set(sk[tr])
            tr_fps = [fpk[j] for j in tr]
            nn = [max(DataStructs.BulkTanimotoSimilarity(fpk[j], tr_fps)) for j in te]
            twin_set = set(fk[tr])
            twin = float(np.mean([f in twin_set for f in fk[te]]))
            rf = RandomForestRegressor(n_estimators=500, random_state=seed, n_jobs=-1).fit(Xk[tr], yk[tr])
            preds = {"RF": rf.predict(Xk[te])}
            svr = SVR(kernel="precomputed", C=5.0, epsilon=0.1).fit(tanimoto_kernel(Xk[tr], Xk[tr]), yk[tr])
            preds["SVR"] = svr.predict(tanimoto_kernel(Xk[te], Xk[tr]))
            for m, p in preds.items():
                rows.append(dict(analysis=tag, seed=seed, split=name, model=m, n_train=len(tr), n_test=len(te),
                                 rmse=float(np.sqrt(mean_squared_error(yk[te], p))), r2=float(r2_score(yk[te], p)),
                                 audit_ratio=aud["leakage_ratio"], test_scaffolds=aud["test_scaffolds"],
                                 test_mol_seen_scaf=float(np.mean([s in seen for s in sk[te]])),
                                 twin_share=twin, nn_median=float(np.median(nn))))
        print(tag, "seed", seed, "完成", f"{time.time() - t0:.0f} 秒")
    return pd.DataFrame(rows)

t0 = time.time()
# 主分析：用 DTA-GNN 的輸出，不另外過濾
res = run(np.ones(len(df0), dtype=bool), "main")
# 事先定好的敏感度分析：去掉 pChEMBL 不在 2–12 之間的紀錄（ChEMBL 裡有單位錯誤的離群值）
ok_range = ((df0.label >= 2) & (df0.label <= 12)).values
print("pChEMBL 落在 2–12 之外的分子：", int((~ok_range).sum()))
res = pd.concat([res, run(ok_range, "range2-12")], ignore_index=True)
res.to_csv("results.csv", index=False)

In [ ]:
# 6. 彙整：平均與範圍（5 個 seed）
agg = res.groupby(["analysis", "split", "model"]).agg(
    rmse_mean=("rmse", "mean"), rmse_min=("rmse", "min"), rmse_max=("rmse", "max"),
    r2_mean=("r2", "mean"), r2_min=("r2", "min"), r2_max=("r2", "max")).round(3)
print(agg)
aud_tab = res[res.model == "RF"].groupby(["analysis", "split"]).agg(
    audit_ratio=("audit_ratio", "mean"), test_mol_seen_scaf=("test_mol_seen_scaf", "mean"),
    twin_share=("twin_share", "mean"), nn_median=("nn_median", "mean"),
    n_train=("n_train", "mean"), n_test=("n_test", "mean")).round(3)
print(aud_tab)

In [ ]:
# 6b. 逐 seed 配對差（骨架 − 隨機）與離群值檢查
print("原始筆數", len(raw), "| 分子", df0.molecule_chembl_id.nunique())
print(df0.sort_values("label").head(3)[["molecule_chembl_id", "label"]].to_string(index=False))
print("pChEMBL 低於 2：", int((df0.label < 2).sum()), "| 高於 12：", int((df0.label > 12).sum()))
for a in ["main", "range2-12"]:
    for m in ["RF", "SVR"]:
        r = res[(res.analysis == a) & (res.model == m)].pivot(index="seed", columns="split", values="rmse")
        diff = r["scaffold"] - r["random"]
        print(a, m, "骨架 - 隨機，逐 seed：", diff.round(3).tolist(), "| 平均", round(diff.mean(), 3),
              "| 骨架較高的 seed 數", int((diff > 0).sum()), "/ 5")

In [ ]:
# 7. 圖
import matplotlib.pyplot as plt
main = res[res.analysis == "main"]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
colors = {"random": "#0b7285", "scaffold": "#e8590c"}
for k, (model, g) in enumerate(main.groupby("model")):
    for j, name in enumerate(["random", "scaffold"]):
        v = g[g.split == name].rmse.values
        x = k + (j - 0.5) * 0.35
        ax[0].scatter([x] * len(v), v, color=colors[name], alpha=.6, s=22)
        ax[0].hlines(v.mean(), x - .13, x + .13, color=colors[name], lw=2.5, label=name if k == 0 else None)
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["RF", "SVR"])
ax[0].set_ylabel("test RMSE (pChEMBL)"); ax[0].legend(); ax[0].set_title("Test error by split")
g = main[main.model == "RF"]
for name in ["random", "scaffold"]:
    v = g[g.split == name].test_mol_seen_scaf.values * 100
    ax[1].bar(name, v.mean(), color=colors[name]); ax[1].scatter([name] * len(v), v, color="k", s=10)
ax[1].set_ylabel("% test molecules with scaffold in train"); ax[1].set_title("Scaffold overlap")
plt.tight_layout(); plt.savefig("fig_split_compare.png", dpi=160); plt.show()

## 怎麼讀這些數字

- 隨機切分和骨架切分的差距，要看**同一個 seed、同一個模型**的配對，不要拿最好的一組去比最差的一組。
- 這個資料集只有一個標靶，所以結論只能說這個標靶、這個切分與這兩個模型。換標靶，差距的大小會變。
- 測試誤差還包含「測試分子本來就比較難」的成分。骨架切分把整組稀有骨架放進測試集，兩種切法的測試分子不是同一批。
- `audit_ratio` 是 DTA-GNN 稽核的結果：測試集骨架**種類**裡，有多少比例也出現在訓練集。
  `test_mol_seen_scaf` 則以**分子數**計算。兩個分母不同，數字會不同。